In [10]:
"""
    Sample a rotation matrix in SO(3) lying within geodesic distance <= theta0
    from the identity.

    Idea:
    - First sample a unit quaternion q uniformly on S^3.
    - A unit quaternion q = (w, x, y, z) represents a rotation in SO(3),
      with rotation angle theta satisfying w = cos(theta / 2).
    - Since q and -q give the same rotation matrix, we fix a canonical
      representative by forcing w >= 0.
    - Then the condition theta <= theta0 is equivalent to
            w >= cos(theta0 / 2).
    - So we reject samples outside that cap on S^3, and convert the accepted
      quaternion into a 3x3 rotation matrix.

    Returns:
        q : accepted unit quaternion (with w >= 0),
        R : corresponding 3x3 rotation matrix in SO(3)
    """
    # theta0 is the radius of the ball around the identity in SO(3),
    # measured using the standard geodesic distance (which equals
    # the rotation angle, always between 0 and pi).

'\n    Sample a rotation matrix in SO(3) lying within geodesic distance <= theta0\n    from the identity.\n\n    Idea:\n    - First sample a unit quaternion q uniformly on S^3.\n    - A unit quaternion q = (w, x, y, z) represents a rotation in SO(3),\n      with rotation angle theta satisfying w = cos(theta / 2).\n    - Since q and -q give the same rotation matrix, we fix a canonical\n      representative by forcing w >= 0.\n    - Then the condition theta <= theta0 is equivalent to\n            w >= cos(theta0 / 2).\n    - So we reject samples outside that cap on S^3, and convert the accepted\n      quaternion into a 3x3 rotation matrix.\n\n    Returns:\n        q : accepted unit quaternion (with w >= 0),\n        R : corresponding 3x3 rotation matrix in SO(3)\n    '

In [4]:
import numpy as np  

In [5]:
#Helper Functions:

def normalize(q: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(q)       
    if n == 0:
        raise ValueError("Zero quaternion cannot be normalized.")
    return q / n

def quat_to_R(q: np.ndarray) -> np.ndarray:
    """
    Map unit quaternion q=(w,x,y,z) to rotation matrix R in SO(3).
    Assumes q is unit-length.
    """
    w, x, y, z = q
    R = np.array([
        [1 - 2*(y*y + z*z),     2*(x*y - z*w),     2*(x*z + y*w)],
        [    2*(x*y + z*w), 1 - 2*(x*x + z*z),     2*(y*z - x*w)],
        [    2*(x*z - y*w),     2*(y*z + x*w), 1 - 2*(x*x + y*y)]
    ], dtype=float)
    return R

def rotation_angle_from_R(R: np.ndarray) -> float:
    """
    Return principal rotation angle theta in [0, pi] from a rotation matrix.
    """
    tr = np.trace(R)
    c = (tr - 1.0) / 2.0
    c = float(np.clip(c, -1.0, 1.0))
    return float(np.arccos(c))

def rotation_angle_from_quat(q: np.ndarray) -> float:
    """
    Return principal rotation angle theta in [0, pi] from a unit quaternion.
    """
    w = float(q[0])
    w = abs(w)  # because q and -q represent same rotation
    w = np.clip(w, -1.0, 1.0)
    return float(2.0 * np.arccos(w))

def check_SO3(R: np.ndarray):
    RtR = R.T @ R
    ortho_err = np.linalg.norm(RtR - np.eye(3), ord='fro')
    detR = float(np.linalg.det(R))
    return (ortho_err <= 1e-8 and abs(detR - 1.0) <= 1e-8)
    

In [6]:
# Sampling:

def sample_bounded_SO3(theta0: float, max_tries: int = 1_000_000) -> tuple[np.ndarray, np.ndarray]:
    """
    Sample Haar-uniform on SO(3) conditioned on rotation angle <= theta0 (0 < theta0 <= pi),
    using rejection sampling on S^3 (unit quaternions), then map to SO(3).

    Returns:
        q (unit quaternion with w >= 0),
        R (3x3 rotation matrix)
    """
    if not (0.0 < theta0 <= np.pi):
        raise ValueError("theta0 must satisfy 0 < theta0 <= pi.")
    c = np.cos(theta0 / 2.0)

    for _ in range(max_tries):
        # Sample uniform on S^3 by normalizing a 4D Gaussian
        q = np.random.normal(size=4)
        q = normalize(q)

        # Fix the double-cover: choose canonical representative with w >= 0
        if q[0] < 0:
            q = -q

        # Bounded condition (cap around north pole in S^3)
        if q[0] >= c:
            R = quat_to_R(q)
            return q, R

    raise RuntimeError("Failed to sample within max_tries. Try increasing max_tries or theta0.")


In [7]:
#Validation:

def check_bounded_and_SO3(q: np.ndarray, R: np.ndarray, theta0: float, tol: float = 1e-10) -> dict:
    """
    Checks:
      1) q is unit and satisfies bounded angle <= theta0 (via quaternion)
      2) R is in SO(3): R^T R ~ I and det(R) ~ 1
      3) The rotation angle from R is also <= theta0 (consistency)

    Returns a dict with diagnostics.
    """
    # Quaternion checks
    q_norm = np.linalg.norm(q)
    theta_q = rotation_angle_from_quat(q)

    # Matrix checks
    RtR = R.T @ R
    ortho_err = np.linalg.norm(RtR - np.eye(3), ord='fro')
    detR = float(np.linalg.det(R))
    theta_R = rotation_angle_from_R(R)

    return {
        "q_norm": q_norm,
        "theta_from_quat": theta_q,
        "theta_from_R": theta_R,
        "bounded_ok_quat": (theta_q <= theta0 + tol),
        "bounded_ok_R": (theta_R <= theta0 + tol),
        "orthogonality_fro_error": ortho_err,
        "detR": detR,
        "SO3_ok": (ortho_err <= 1e-8 and abs(detR - 1.0) <= 1e-8),
    }

In [8]:
if __name__ == "__main__":
    n = 2 # number of matrices
    theta0 = np.deg2rad(20.0)  # bound: 20 degrees
    q, R = sample_bounded_SO3(theta0)

    info = check_bounded_and_SO3(q, R, theta0)
    print("Quaternion q =", q)
    print("Rotation matrix R =\n", R)
    print("\nChecks:")
    for k, v in info.items():
        print(f"  {k}: {v}")

    SO3_matrices = []
    for i in range(n):
        _, R = sample_bounded_SO3(theta0)
        print(R)
        print(check_SO3(R))
        SO3_matrices.append(R)
        

Quaternion q = [0.98750612 0.07586453 0.1068096  0.08756675]
Rotation matrix R =
 [[ 0.96184755 -0.15673928  0.22423669]
 [ 0.18915152  0.97315328 -0.13112744]
 [-0.19766385  0.16853932  0.96567256]]

Checks:
  q_norm: 1.0
  theta_from_quat: 0.3164803729991762
  theta_from_R: 0.31648037299917664
  bounded_ok_quat: True
  bounded_ok_R: True
  orthogonality_fro_error: 4.889017563180917e-17
  detR: 0.9999999999999999
  SO3_ok: True
[[ 0.98937223 -0.14366581  0.02242161]
 [ 0.13924204  0.98052399  0.13850758]
 [-0.04188373 -0.13391352  0.99010753]]
True
[[ 0.97695479  0.06066987  0.20464239]
 [-0.03742378  0.99258931 -0.11561104]
 [-0.21013996  0.10528827  0.97198538]]
True


In [ ]:
as